# AI-Driven Loan Analytics Platform

## Data Cleaning Notebook

This notebook creates a cleaner version of the raw loan dataset while keeping the data faithful to the original source.

The goal is not to make the dataset model-ready yet. The goal is to prepare a clean base file for later analysis, feature engineering, and modeling.

Main principles:

- keep the raw dataset unchanged;
- clean structure and formats;
- avoid inventing missing values;
- protect target-like columns;
- keep the cleaning logic easy to audit.

## 1. Import libraries

We use a small set of common Python libraries. Pandas handles tabular data, NumPy helps with missing values and numeric checks, and `Path` keeps file paths clean.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 2. Configuration

The raw input path and cleaned output path are defined in one place. This makes the notebook easy to reuse with another loan dataset.

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'loan_dataset_investor.xlsx'
DATA_SHEET = 0
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'data_cleaned.csv'

TARGET_CANDIDATES = [
    'has_default_within_12_months',
    'loan_default',
    'default',
    'is_default',
    'target',
    'loan_status',
    'status',
    'repayment_status',
]

KNOWN_BINARY_COLUMNS = [
    'is_early_repaid_within_14_days',
    'has_default_within_12_months',
    'is_default',
]

print(f'Raw data path: {DATA_PATH}')
print(f'Cleaned data output path: {OUTPUT_PATH}')

Raw data path: c:\Users\aitmo\financial-loan-analytics\data\raw\loan_dataset_investor.xlsx
Cleaned data output path: c:\Users\aitmo\financial-loan-analytics\data\processed\data_cleaned.csv


## 3. Helper functions

These helper functions make the notebook modular and easier to review. Each function performs one clear cleaning or validation task.

In [3]:
def load_dataset(path, sheet_name=0):
    """Load a CSV or Excel dataset without modifying the source file."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f'Dataset not found: {path}')

    if path.suffix.lower() in ['.xlsx', '.xls']:
        return pd.read_excel(path, sheet_name=sheet_name)

    if path.suffix.lower() == '.csv':
        return pd.read_csv(path)

    raise ValueError(f'Unsupported file type: {path.suffix}')


def clean_column_name(column_name):
    """Convert a column name to a simple snake_case format."""
    clean_name = str(column_name).strip().lower()
    clean_name = re.sub(r'[^0-9a-zA-Z]+', '_', clean_name)
    clean_name = clean_name.strip('_')
    return clean_name or 'column'


def make_unique_column_names(columns):
    """Avoid duplicate column names after standardization."""
    seen = {}
    unique_columns = []

    for column in columns:
        count = seen.get(column, 0)
        unique_columns.append(column if count == 0 else f'{column}_{count + 1}')
        seen[column] = count + 1

    return unique_columns


def detect_date_columns(dataframe):
    """Find date-like columns using dtype and column names."""
    date_columns = []

    for column in dataframe.columns:
        is_datetime = pd.api.types.is_datetime64_any_dtype(dataframe[column])
        has_date_name = 'date' in column or column.endswith('_at')

        if is_datetime or has_date_name:
            date_columns.append(column)

    return date_columns


def summarize_dataframe(dataframe):
    """Create a compact quality summary for before/after checks."""
    return pd.DataFrame(
        {
            'metric': [
                'rows',
                'columns',
                'duplicate_rows',
                'total_missing_values',
                'columns_with_missing_values',
            ],
            'value': [
                len(dataframe),
                dataframe.shape[1],
                int(dataframe.duplicated().sum()),
                int(dataframe.isna().sum().sum()),
                int((dataframe.isna().sum() > 0).sum()),
            ],
        }
    )


def binary_value_report(dataframe, columns):
    """Check whether selected columns contain only 0, 1, and missing values."""
    rows = []

    for column in columns:
        if column not in dataframe.columns:
            continue

        non_missing_values = set(dataframe[column].dropna().unique().tolist())
        is_binary = non_missing_values.issubset({0, 1, 0.0, 1.0})

        rows.append(
            {
                'column': column,
                'non_missing_values': sorted(non_missing_values),
                'missing_values': int(dataframe[column].isna().sum()),
                'is_binary_0_1': is_binary,
            }
        )

    return pd.DataFrame(rows)

## 4. Load the raw dataset

We load the raw dataset and create a working copy. The original Excel file remains unchanged in `data/raw/`.

In [4]:
raw_df = load_dataset(DATA_PATH, DATA_SHEET)
df = raw_df.copy()

display(summarize_dataframe(raw_df))
display(raw_df.head())

,metric,value
0,rows,737889
1,columns,31
2,duplicate_rows,0
3,total_missing_values,4712422
4,columns_with_missing_values,23


,loan_id,country,loan_issued_at,early_repaid_at,is_early_repaid_within_14_days,issued_amount,loan_status,loan_last_recorded_action_date_local,initial_interest_rate,nr_of_payments,principal_balance,principal_debt,principal_paid_total,interest_paid_total,extra_interest_paid_total,late_fee_paid_total,maintenance_fee_paid_total,is_default,next_payment_nr,next_payment_date_local,debt_occured_date_local,days_past_due_principal,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,projected_npv_return,customer_risk_rating
0,0B3B642A-F923-49AD-84ED-B2A700BF1D0A,Finland,2025-03-21 11:45:56.040,2025-04-04 10:50:01.137,1.00,310.00,Repaid,2025-04-04,0.18,1,0.00,0.00,310.00,1.99,0.00,0.00,0.00,0.00,0.00,NaT,NaT,NaN,NaN,0.00,Cancelled,311.99,12,"2,068.68",0.00,0.02,F
1,4FD1927C-0E7F-4BF1-AED0-B268011E3E85,Finland,2025-01-17 17:27:25.447,NaT,NaN,414.00,Active,NaT,0.18,120,301.49,0.00,112.51,70.83,0.00,0.00,3.27,0.00,11.00,2026-05-08,NaT,NaN,NaN,15.00,Active,186.61,18,"1,469.50",0.00,0.12,B
2,7362E260-54DC-4798-8848-B18C00A59FCB,Finland,2024-06-11 10:04:52.427,2025-01-02 12:07:32.807,0.00,518.00,Repaid,2025-01-02,0.19,8,0.00,0.00,518.00,51.60,0.00,0.00,0.00,0.00,0.00,NaT,NaT,NaN,NaN,6.00,Paid Up,569.60,60,"4,513.20",0.00,0.12,B
3,8C7E5FB6-A673-4148-BCCF-B27500B0368E,Finland,2025-01-30 10:52:12.657,NaT,NaN,"3,731.00",Active,NaT,0.18,75,"3,305.87",0.00,425.13,724.93,0.00,0.00,22.63,0.00,14.00,2026-05-04,NaT,NaN,NaN,15.00,Active,"1,172.69",60,"1,301.00",0.00,0.10,C
4,BE865034-6041-4855-AB42-B196014FA0E8,Netherlands,2024-06-21 20:48:23.787,NaT,NaN,"3,432.00",Active,NaT,0.11,38,"2,371.82",0.00,"1,060.18",549.66,0.29,0.00,60.65,0.00,1.00,2026-05-01,NaT,NaN,NaN,22.00,Active,"1,670.78",60,"5,194.70",0.00,0.11,A


## 5. Pre-cleaning quality check

Before changing anything, we inspect missing values and duplicates. This gives us a baseline for the cleaning process.

In [5]:
missing_before = (
    raw_df.isna()
    .sum()
    .to_frame('missing_values')
    .assign(missing_percentage=lambda table: table['missing_values'] / len(raw_df) * 100)
    .sort_values('missing_values', ascending=False)
)

display(Markdown('### Missing values before cleaning'))
display(missing_before[missing_before['missing_values'] > 0].head(30))

duplicate_rows_before = int(raw_df.duplicated().sum())
print(f'Duplicate rows before cleaning: {duplicate_rows_before:,}')

### Missing values before cleaning

,missing_values,missing_percentage
months_in_default,637048,86.33
is_early_repaid_within_14_days,525405,71.20
early_repaid_at,525405,71.20
debt_occured_date_local,525129,71.17
days_past_due_principal,525129,71.17
next_payment_date_local,428662,58.09
loan_last_recorded_action_date_local,400285,54.25
customer_risk_rating,339601,46.02
combined_income,273156,37.02
projected_npv_return,272284,36.90


Duplicate rows before cleaning: 0


## 6. Standardize column names

Clean column names reduce future coding errors. We convert names to `snake_case` and make sure every column name is unique.

In [6]:
original_columns = list(df.columns)
clean_columns = [clean_column_name(column) for column in original_columns]
df.columns = make_unique_column_names(clean_columns)

column_mapping = pd.DataFrame(
    {
        'original_column': original_columns,
        'clean_column': df.columns,
    }
)

changed_columns = column_mapping[
    column_mapping['original_column'] != column_mapping['clean_column']
]

if changed_columns.empty:
    print('No column names needed changes.')
else:
    display(changed_columns)

No column names needed changes.


## 7. Clean text values

For text columns, we remove extra spaces and convert empty text tokens into missing values.

We do not replace missing categories with `Unknown` here, because that is a modeling decision and can hide useful missingness patterns.

In [7]:
text_columns = df.select_dtypes(include=['object', 'string']).columns.tolist()
missing_tokens = {'', 'nan', 'none', 'null', 'n/a', 'na'}

for column in text_columns:
    cleaned_text = df[column].astype('string').str.strip()
    df[column] = cleaned_text.mask(cleaned_text.str.lower().isin(missing_tokens))

print(f'Text columns cleaned: {len(text_columns)}')
display(pd.DataFrame({'text_column': text_columns}))

Text columns cleaned: 5


,text_column
0,loan_id
1,country
2,loan_status
3,loan_status_risk
4,customer_risk_rating


## 8. Convert date columns

Date fields are important for loan lifecycle analysis. We convert date-like columns to datetime format and keep invalid or missing dates as missing values.

In [8]:
date_columns = detect_date_columns(df)

for column in date_columns:
    df[column] = pd.to_datetime(df[column], errors='coerce')

print(f'Date columns converted: {len(date_columns)}')
display(pd.DataFrame({'date_column': date_columns}))

Date columns converted: 5


,date_column
0,loan_issued_at
1,early_repaid_at
2,loan_last_recorded_action_date_local
3,next_payment_date_local
4,debt_occured_date_local


## 9. Validate binary columns

Some important loan-risk columns are binary indicators. We verify that they contain only `0`, `1`, and missing values.

Missing binary values are preserved. We do not guess whether a missing value means `0` or `1`.

In [9]:
binary_report = binary_value_report(df, KNOWN_BINARY_COLUMNS)
display(binary_report)

invalid_binary_columns = binary_report.loc[
    ~binary_report['is_binary_0_1'], 'column'
].tolist()

if invalid_binary_columns:
    print('Warning: Some binary columns contain values outside 0/1:')
    print(invalid_binary_columns)
else:
    print('All checked binary columns contain only 0, 1, and missing values.')

# Convert valid binary indicators to nullable integers.
# This keeps 0/1 clean while preserving missing values as <NA>.
for column in binary_report.loc[binary_report['is_binary_0_1'], 'column']:
    df[column] = df[column].astype('Int64')

,column,non_missing_values,missing_values,is_binary_0_1
0,is_early_repaid_within_14_days,"[0.0, 1.0]",525405,True
1,has_default_within_12_months,"[0.0, 1.0]",174379,True
2,is_default,"[0.0, 1.0]",7713,True


All checked binary columns contain only 0, 1, and missing values.


## 10. Remove exact duplicate rows

Exact duplicate rows can overstate portfolio size and distort risk metrics. We remove only fully identical rows.

In [10]:
duplicate_rows_before = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

print(f'Duplicate rows removed: {duplicate_rows_before:,}')
print(f'Rows after duplicate removal: {len(df):,}')

Duplicate rows removed: 0
Rows after duplicate removal: 737,889


## 11. Preserve missing numerical values

We intentionally do not fill missing numerical values in this cleaning notebook.

Why? In credit-risk data, missing values can carry business meaning. For example, a missing default date may mean the loan has not defaulted, while a missing income value may mean the borrower did not report income.

Imputation will be handled later during feature engineering or modeling, where we can use safer rules and validation.

In [11]:
numeric_missing = (
    df.select_dtypes(include=[np.number])
    .isna()
    .sum()
    .to_frame('missing_values')
    .assign(missing_percentage=lambda table: table['missing_values'] / len(df) * 100)
    .sort_values('missing_values', ascending=False)
)

display(numeric_missing[numeric_missing['missing_values'] > 0])

print('No numerical imputation was applied in this notebook.')

,missing_values,missing_percentage
months_in_default,637048,86.33
is_early_repaid_within_14_days,525405,71.20
days_past_due_principal,525129,71.17
combined_income,273156,37.02
projected_npv_return,272284,36.90
has_default_within_12_months,174379,23.63
months_on_book,8806,1.19
maintenance_fee_paid_total,7713,1.05
principal_balance,7713,1.05
next_payment_nr,7713,1.05


No numerical imputation was applied in this notebook.


## 12. Post-cleaning quality check

We compare the cleaned dataset against the original baseline. The number of rows should only change if exact duplicate rows were removed.

In [12]:
display(Markdown('### Cleaned dataset summary'))
display(summarize_dataframe(df))

missing_after = (
    df.isna()
    .sum()
    .to_frame('missing_values')
    .assign(missing_percentage=lambda table: table['missing_values'] / len(df) * 100)
    .sort_values('missing_values', ascending=False)
)

display(Markdown('### Missing values after cleaning'))
display(missing_after[missing_after['missing_values'] > 0].head(30))

print(f'Final dataset shape: {df.shape[0]:,} rows and {df.shape[1]:,} columns')

### Cleaned dataset summary

,metric,value
0,rows,737889
1,columns,31
2,duplicate_rows,0
3,total_missing_values,4712422
4,columns_with_missing_values,23


### Missing values after cleaning

,missing_values,missing_percentage
months_in_default,637048,86.33
is_early_repaid_within_14_days,525405,71.20
early_repaid_at,525405,71.20
debt_occured_date_local,525129,71.17
days_past_due_principal,525129,71.17
next_payment_date_local,428662,58.09
loan_last_recorded_action_date_local,400285,54.25
customer_risk_rating,339601,46.02
combined_income,273156,37.02
projected_npv_return,272284,36.90


Final dataset shape: 737,889 rows and 31 columns


## 13. Save the cleaned dataset

The cleaned file is saved separately from the raw file. This keeps the project reproducible: raw data remains in `data/raw/`, and cleaned data goes to `data/processed/`.

In [13]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f'Cleaned dataset saved to: {OUTPUT_PATH}')

Cleaned dataset saved to: c:\Users\aitmo\financial-loan-analytics\data\processed\data_cleaned.csv


## 14. Final notes

This cleaned dataset is now a safer base for the next steps.

Recommended next steps:

- create a feature engineering notebook;
- define the prediction objective clearly;
- decide which columns are available at prediction time;
- handle missing values inside a modeling pipeline;
- avoid target leakage from post-loan or default-status variables.